In [1]:
%pip install python-crfsuite
import nltk
nltk.download('conll2002')
from nltk.corpus import conll2002

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: c:\Users\maxvi\.pyenv\pyenv-win\versions\3.12.7\python.exe -m pip install --upgrade pip
[nltk_data] Downloading package conll2002 to
[nltk_data]     C:\Users\maxvi\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2002 is already up-to-date!


In [2]:
import os

models_dir = os.path.join(os.getcwd(), 'models')
os.makedirs(models_dir, exist_ok=True)

In [3]:
esp_train = conll2002.iob_sents('esp.train') # Train, ned.train => Neerlandès
esp_val = conll2002.iob_sents('esp.testa') # Dev
esp_test = conll2002.iob_sents('esp.testb') # Test

In [4]:
ned_train = conll2002.iob_sents('ned.train') # Train, ned.train => Neerlandès
ned_val = conll2002.iob_sents('ned.testa') # Dev
ned_test = conll2002.iob_sents('ned.testb')

In [5]:
def prepare_data(sents):
    return [[(word,bio) for word, pos, bio in sent] for sent in sents]

In [6]:
esp_train_net = prepare_data(esp_train)
esp_val_net = prepare_data(esp_val)
esp_test_net = prepare_data(esp_test)
ned_train_net = prepare_data(ned_train)
ned_val_net = prepare_data(ned_val)
ned_test_net = prepare_data(ned_test)

In [7]:
def bio_to_io(sentence):
    new_sentence = []
    
    for word, tag in sentence:
        if tag.startswith("B-") or tag.startswith("I-"):
            new_tag = "I-" + tag[2:]
        else:
            new_tag = tag  # "O"
            
        new_sentence.append((word, new_tag))
    
    return new_sentence

In [8]:
def bio_to_biow(sentence):
    new_sentence = []
    i = 0
    n = len(sentence)
    
    while i < n:
        word, tag = sentence[i]
        
        if tag.startswith("B-"):
            entity_type = tag[2:]
            j = i + 1
            
            # trobar final de l'entitat
            while j < n and sentence[j][1] == f"I-{entity_type}":
                j += 1
            
            length = j - i
            
            if length == 1:
                new_sentence.append((word, f"W-{entity_type}"))
            else:
                new_sentence.append((word, f"B-{entity_type}"))
                for k in range(i+1, j):
                    new_sentence.append((sentence[k][0], f"I-{entity_type}"))
            
            i = j
        
        else:
            new_sentence.append((word, tag))
            i += 1
    
    return new_sentence

In [9]:
def bio_to_bioew(sentence):
    new_sentence = []
    i = 0
    n = len(sentence)
    
    while i < n:
        word, tag = sentence[i]
        
        if tag.startswith("B-"):
            entity_type = tag[2:]
            j = i + 1
            
            # trobar final de l'entitat
            while j < n and sentence[j][1] == f"I-{entity_type}":
                j += 1
            
            length = j - i
            
            if length == 1:
                new_sentence.append((word, f"W-{entity_type}"))
            else:
                # primer
                new_sentence.append((word, f"B-{entity_type}"))
                
                # mig
                for k in range(i+1, j-1):
                    new_sentence.append((sentence[k][0], f"I-{entity_type}"))
                
                # últim
                last_word = sentence[j-1][0]
                new_sentence.append((last_word, f"E-{entity_type}"))
            
            i = j
        
        else:
            new_sentence.append((word, tag))
            i += 1
    
    return new_sentence

In [10]:
from typing import List
import string

class FeatureFunc:
    def __init__(self, use_casing=True, use_punct=True, use_nums=True, 
                 use_suffix=True, use_prefix=True, use_length=True, 
                 use_context=True, use_pos=True):
        self.use_casing  = use_casing
        self.use_punct   = use_punct
        self.use_nums    = use_nums
        self.use_suffix  = use_suffix
        self.use_prefix  = use_prefix
        self.use_length  = use_length
        self.use_context = use_context
        self.use_pos     = use_pos

    def __call__(self, tokens: List[str], idx: int) -> List[str]:
        word = tokens[idx]

        # --- OBLIGATÒRIES ---

        # Paraula actual (sempre)
        features = ['word=' + word, 'lower=' + word.lower()]

        # És majúscula?
        if self.use_casing:
            features.append('isupper=%s'  % word.isupper())
            features.append('istitle=%s'  % word.istitle())
            features.append('islower=%s'  % word.islower())

        # Té signes de puntuació?
        if self.use_punct:
            features.append('ispunct=%s'   % all(c in string.punctuation for c in word))
            features.append('haspunct=%s'  % any(c in string.punctuation for c in word))

        # Té números?
        if self.use_nums:
            features.append('isdigit=%s'   % word.isdigit())
            features.append('hasdigit=%s'  % any(c.isdigit() for c in word))

        # Sufixos
        if self.use_suffix:
            features.append('suffix1=' + word[-1:])
            features.append('suffix2=' + word[-2:])
            features.append('suffix3=' + word[-3:])

        # --- OPCIONALS ---

        # Prefixos
        if self.use_prefix:
            features.append('prefix1=' + word[:1])
            features.append('prefix2=' + word[:2])
            features.append('prefix3=' + word[:3])

        # Longitud
        if self.use_length:
            features.append('len=%d' % len(word))

        # Context (paraula anterior i següent)
        if self.use_context:
            if idx > 0:
                prev = tokens[idx-1]
                features.append('prev='        + prev.lower())
                features.append('prev_istitle=%s' % prev.istitle())
                features.append('prev_isupper=%s' % prev.isupper())
            else:
                features.append('BOS')
            if idx < len(tokens) - 1:
                next_ = tokens[idx+1]
                features.append('next='        + next_.lower())
                features.append('next_istitle=%s' % next_.istitle())
                features.append('next_isupper=%s' % next_.isupper())
            else:
                features.append('EOS')

        return features

In [11]:
def convert_coding(dataset, coding):
    if coding == "BIO":
        return dataset
    elif coding == "IO":
        return [bio_to_io(sent) for sent in dataset]
    elif coding == "BIOW":
        return [bio_to_biow(sent) for sent in dataset]
    elif coding == "BIOEW":
        return [bio_to_bioew(sent) for sent in dataset]
    else:
        raise ValueError(f"Coding desconegut: {coding}")

In [12]:
def evaluate(model, data):
    tp, fp, fn = 0, 0, 0

    for sent in data:
        tokens  = [word for word, tag in sent]
        gold    = [tag  for word, tag in sent]
        pred    = [tag  for word, tag in model.tag(tokens)]

        for g, p in zip(gold, pred):
            g_is_entity = g != 'O'
            p_is_entity = p != 'O'

            if g_is_entity and p_is_entity:
                if g == p:
                    tp += 1   # mateixa etiqueta (ex: B-PER == B-PER)
                else:
                    fp += 1   # entitat però etiqueta diferent (ex: B-PER vs I-PER)
                    fn += 1   # la gold no s'ha trobat correctament
            elif p_is_entity and not g_is_entity:
                fp += 1       # prediu entitat on no n'hi ha
            elif g_is_entity and not p_is_entity:
                fn += 1       # no prediu entitat on n'hi havia

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {'precision': precision, 'recall': recall, 'f1': f1}

In [15]:
feature_configs = [
    ('baseline',        FeatureFunc(use_casing=False, use_nums=False, use_prefix=False,
                                    use_suffix=False, use_context=False, use_length=False)),
    ('casing_only',     FeatureFunc(use_nums=False, use_prefix=False,
                                    use_suffix=False, use_context=False, use_length=False)),
    ('full',            FeatureFunc()),  # tot a True per defecte
    ('no_context',      FeatureFunc(use_context=False)),
    ('prefix_suffix',   FeatureFunc(use_casing=False, use_nums=False,
                                    use_context=False, use_length=False)),
]

import pandas as pd
results = []

for coding in ['BIO', 'IO', 'BIOW', 'BIOEW']:
    train_esp = convert_coding(esp_train_net, coding)
    train_ned = convert_coding(ned_train_net, coding)
    val_esp   = convert_coding(esp_val_net,   coding)
    val_ned   = convert_coding(ned_val_net,   coding)

    for feat_name, feat_func in feature_configs:
        for lang, train, val in [('ESP', train_esp, val_esp),
                                  ('NED', train_ned, val_ned)]:

            model = nltk.tag.CRFTagger(feature_func=feat_func)
            model.train(train, os.path.join(models_dir, f'crf_{coding}_{feat_name}_{lang}.mdl'))

            metrics = evaluate(model, val)
            print(f"{coding} | {feat_name} | {lang} => "
                  f"P={metrics['precision']:.4f} "
                  f"R={metrics['recall']:.4f} "
                  f"F1={metrics['f1']:.4f}")
            results.append({'coding': coding, 'features': feat_name,
                            'lang': lang, **metrics})

df = pd.DataFrame(results)
print(df.pivot_table(index=['coding', 'features'], columns='lang', values='f1'))

BIO | baseline | ESP => P=0.7571 R=0.4491 F1=0.5637
BIO | baseline | NED => P=0.8536 R=0.3061 F1=0.4507
BIO | casing_only | ESP => P=0.6517 R=0.6701 F1=0.6608
BIO | casing_only | NED => P=0.6502 R=0.5967 F1=0.6223
BIO | full | ESP => P=0.7748 R=0.7560 F1=0.7653
BIO | full | NED => P=0.7680 R=0.7254 F1=0.7461
BIO | no_context | ESP => P=0.7255 R=0.7202 F1=0.7229
BIO | no_context | NED => P=0.7434 R=0.7011 F1=0.7216
BIO | prefix_suffix | ESP => P=0.7209 R=0.7152 F1=0.7181
BIO | prefix_suffix | NED => P=0.7452 R=0.6820 F1=0.7122
IO | baseline | ESP => P=0.7244 R=0.4276 F1=0.5378
IO | baseline | NED => P=0.8185 R=0.2986 F1=0.4376
IO | casing_only | ESP => P=0.6592 R=0.6627 F1=0.6610
IO | casing_only | NED => P=0.6722 R=0.6222 F1=0.6463
IO | full | ESP => P=0.7776 R=0.7584 F1=0.7679
IO | full | NED => P=0.7767 R=0.7342 F1=0.7549
IO | no_context | ESP => P=0.7116 R=0.7108 F1=0.7112
IO | no_context | NED => P=0.7387 R=0.6879 F1=0.7124
IO | prefix_suffix | ESP => P=0.7161 R=0.7102 F1=0.7131
IO

In [18]:
import os
import nltk

for lang in ['ESP', 'NED']:
    best = df[df['lang'] == lang].sort_values('f1', ascending=False).iloc[0]
    coding = best['coding']
    feat_name = best['features']

    print(f"\nMillor configuració per {lang}: coding={coding}, features={feat_name}")

    # Path del model ja entrenat (mateix naming que abans!)
    model_path = os.path.join(models_dir, f'crf_{coding}_{feat_name}_{lang}.mdl')

    best_feat_func = dict(feature_configs)[feat_name]

    model = nltk.tag.CRFTagger(feature_func=best_feat_func)
    model.set_model_file(model_path)

    # Preparem només TEST (no cal train aquí)
    test_data = convert_coding(
        esp_test_net if lang == 'ESP' else ned_test_net,
        coding
    )

    # Avaluació
    metrics = evaluate(model, test_data)

    print(f"Resultats en TEST per {lang}: "
          f"P={metrics['precision']:.4f} "
          f"R={metrics['recall']:.4f} "
          f"F1={metrics['f1']:.4f}")


Millor configuració per ESP: coding=IO, features=full
Resultats en TEST per ESP: P=0.8062 R=0.7852 F1=0.7956

Millor configuració per NED: coding=IO, features=full
Resultats en TEST per NED: P=0.7773 R=0.7405 F1=0.7584


In [20]:
import os
import pandas as pd
import nltk

final_results = []

for lang in ['ESP', 'NED']:
    # Millor configuració segons VALIDACIÓ
    best = df[df['lang'] == lang].sort_values('f1', ascending=False).iloc[0]
    coding = best['coding']
    feat_name = best['features']

    print(f"\nMillor configuració per {lang}: coding={coding}, features={feat_name}")

    # Path del model ja entrenat
    model_path = os.path.join(models_dir, f'crf_{coding}_{feat_name}_{lang}.mdl')

    best_feat_func = dict(feature_configs)[feat_name]

    model = nltk.tag.CRFTagger(feature_func=best_feat_func)
    model.set_model_file(model_path)


    # Preparem dades TEST amb el mateix coding
    test_data = convert_coding(
        esp_test_net if lang == 'ESP' else ned_test_net,
        coding
    )

    # Avaluació TEST
    test_metrics = evaluate(model, test_data)

    print(f"TEST {lang}: "
          f"P={test_metrics['precision']:.4f} "
          f"R={test_metrics['recall']:.4f} "
          f"F1={test_metrics['f1']:.4f}")

    # Guardem comparació
    final_results.append({
        'lang': lang,
        'coding': coding,
        'features': feat_name,
        'val_f1': best['f1'],
        'test_f1': test_metrics['f1'],
        'val_precision': best['precision'],
        'test_precision': test_metrics['precision'],
        'val_recall': best['recall'],
        'test_recall': test_metrics['recall'],
    })

# Taula final
final_df = pd.DataFrame(final_results)
print("\nComparació VAL vs TEST:")
print(final_df)


Millor configuració per ESP: coding=IO, features=full
TEST ESP: P=0.8062 R=0.7852 F1=0.7956

Millor configuració per NED: coding=IO, features=full
TEST NED: P=0.7773 R=0.7405 F1=0.7584

Comparació VAL vs TEST:
  lang coding features    val_f1   test_f1  val_precision  test_precision  \
0  ESP     IO     full  0.767913  0.795572       0.777642        0.806216   
1  NED     IO     full  0.754879  0.758449       0.776702        0.777251   

   val_recall  test_recall  
0    0.758425     0.785206  
1    0.734249     0.740535  


In [21]:
from collections import Counter

def get_confusions(model, data):
    confusions = Counter()

    for sent in data:
        tokens = [w for w, t in sent]
        true_tags = [t for w, t in sent]
        pred_tags = model.tag(tokens)
        pred_tags = [t for w, t in pred_tags]

        for t_true, t_pred in zip(true_tags, pred_tags):
            if t_true != t_pred:
                confusions[(t_true, t_pred)] += 1

    return confusions

# Exemple amb millor model ESP
lang = 'ESP'
best = df[df['lang'] == lang].sort_values('f1', ascending=False).iloc[0]
model_path = os.path.join(models_dir, f"crf_{best['coding']}_{best['features']}_{lang}.mdl")

best_feat_func = dict(feature_configs)[feat_name]

model = nltk.tag.CRFTagger(feature_func=best_feat_func)
model.set_model_file(model_path)

test_data = convert_coding(esp_test_net, best['coding'])

conf = get_confusions(model, test_data)

print("Errors més comuns:")
for (true, pred), count in conf.most_common(15):
    print(f"{true} → {pred}: {count}")

Errors més comuns:
I-MISC → O: 185
I-LOC → I-ORG: 182
I-ORG → I-LOC: 166
I-MISC → I-ORG: 126
I-ORG → I-MISC: 117
O → I-MISC: 88
I-ORG → I-PER: 87
I-ORG → O: 86
I-LOC → I-PER: 75
O → I-ORG: 68
I-LOC → O: 59
I-PER → I-LOC: 59
I-MISC → I-LOC: 56
I-MISC → I-PER: 32
I-PER → I-ORG: 30


In [22]:
def show_errors(model, data, max_sent=5):
    shown = 0

    for sent in data:
        tokens = [w for w, t in sent]
        true_tags = [t for w, t in sent]
        pred = model.tag(tokens)
        pred_tags = [t for w, t in pred]

        if true_tags != pred_tags:
            print("\nFrase:")
            print(" ".join(tokens))

            print("\nTRUE:")
            print(true_tags)

            print("\nPRED:")
            print(pred_tags)

            shown += 1
            if shown >= max_sent:
                break

show_errors(model, test_data, max_sent=5)


Frase:
Las reservas " on line " de billetes aéreos a través de Internet aumentaron en España un 300 por ciento en el primer trimestre de este año con respecto al mismo período de 1999 , aseguró hoy Iñigo García Aranda , responsable de comunicación de Savia Amadeus .

TRUE:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-MISC', 'O', 'O', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'I-ORG', 'I-ORG', 'O']

PRED:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-MISC', 'O', 'O', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'I-LOC', 'I-LOC', 'O']

Frase:
García Aranda presentó a la prensa el sistema Amadeus , que utilizan la mayor parte de las agencias de viajes españolas para reservar billetes de avión o tren , así como 

In [23]:
from collections import defaultdict

def strip_prefix(tag):
    if tag == 'O':
        return 'O'
    return tag.split('-')[-1]

def entity_level_errors(model, data):
    errors = defaultdict(int)
    totals = defaultdict(int)

    for sent in data:
        tokens = [w for w, t in sent]
        true_tags = [strip_prefix(t) for w, t in sent]
        pred_tags = [strip_prefix(t) for w, t in model.tag(tokens)]

        for t_true, t_pred in zip(true_tags, pred_tags):
            totals[t_true] += 1
            if t_true != t_pred:
                errors[t_true] += 1

    return errors, totals

errors, totals = entity_level_errors(model, test_data)

print("Error rate per entitat:")
for ent in totals:
    if ent != 'O':
        err_rate = errors[ent] / totals[ent]
        print(f"{ent}: {err_rate:.3f}")

Error rate per entitat:
LOC: 0.244
ORG: 0.182
MISC: 0.445
PER: 0.093


### Fer la classe de Feature Funciton + Mirar accuracy en funció de entitats i recall i tal